In [ ]:
import pandas as pd
import json
import os
from openai import OpenAI
from tqdm import tqdm  # 进度条库，需 pip install tqdm
import time
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

# ================= 配置区域 =================
API_KEY = os.getenv("API_KEY")  # 从环境变量读取
BASE_URL = os.getenv("BASE_URL", "https://dashscope.aliyuncs.com/compatible-mode/v1")
MODEL_NAME = os.getenv("MODEL_NAME", "qwen-plus-2025-12-01")

# 验证 API Key 是否存在
if not API_KEY:
    raise ValueError("请在 .env 文件中设置 API_KEY")

In [ ]:
# 文件夹路径
INPUT_FOLDER = "原始数据"       # 存放多个excel文件的文件夹路径
OUTPUT_FOLDER = "分类结果"      # 结果输出路径
CLASS_FILE = "职业分类.xlsx"          # 分类表文件
# ===========================================


def get_standard_category(descriptions, class_info_str, restricted=False):
    """
    调用大模型进行归类
    restricted=True 时，Prompt会强制限制在非从业人员分类中
    """
    client = OpenAI(api_key=API_KEY, base_url=BASE_URL)
    
    constraint_text = ""
    if restricted:
        constraint_text = """
        【重要限制】
        这组数据通过视觉无法确认职业，属于非从业人员场景。
        请**仅从以下三个类别中选择**，严禁归入其他职业类：
        - 0 (学生/儿童)
        - 9 (家庭角色，如母亲、父亲、祖辈)
        - 10 (其他非从业人员，或无法判断的非职业身份)
        """
    else:
        constraint_text = "请根据描述归入最合适的类别（0-10类）。如果描述的是学生、儿童或家庭角色，请优先归入0或9类。"

    prompt = f"""
    任务：请将以下人物描述归类到《职业分类大典》的对应类别中。
    
    参考分类标准：
    {class_info_str}
    
    {constraint_text}
    
    待分类的描述列表：
    {json.dumps(descriptions, ensure_ascii=False)}
    
    要求：
    1. 严格输出JSON格式，Key为原描述，Value为对应的"代码"（数字）。
    2. 不要输出任何解释性文字。
    
    输出示例：
    {{"正在讲课的老师": 2, "背书包的小学生": 0, "做饭的母亲": 9}}
    """

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            response_format={"type": "json_object"}
        )
        content = response.choices[0].message.content
        return json.loads(content)
    except Exception as e:
        print(f"API调用出错: {e}")
        return {}

def process_single_file(file_path, output_path, df_class):
    print(f"正在处理文件: {os.path.basename(file_path)}")
    
    # 读取文件
    try:
        if file_path.endswith('.xlsx'):
            df_data = pd.read_excel(file_path)
        else:
            df_data = pd.read_csv(file_path)
    except Exception as e:
        print(f"读取失败，跳过: {e}")
        return

    # === 预处理：构建分类参考信息字符串 ===
    # 1. 全量分类信息（用于已知职业）
    class_info_full = ""
    for _, row in df_class.iterrows():
        class_info_full += f"代码[{row['代码']}] {row['分类名称']}: {str(row['分类内容'])[:30]}...\n"
    
    # 2. 限制分类信息（用于未知职业）
    # 只提取代码为 0, 9, 10 的行
    restricted_codes = [0, 9, 10]
    class_info_restricted = ""
    for _, row in df_class[df_class['代码'].isin(restricted_codes)].iterrows():
        class_info_restricted += f"代码[{row['代码']}] {row['分类名称']}: {str(row['分类内容'])[:30]}...\n"

    # === 双轨制处理 ===
    mapping_dict = {}

    # 轨道A：Profession 不为 "未知" 的样本 -> 归类 Profession -> 使用全量分类表
    known_mask = df_data['profession'] != '未知'
    # 提取需要归类的词（排除空值）
    descs_known = df_data.loc[known_mask, 'profession'].dropna().unique().tolist()
    
    if descs_known:
        print(f"  - 轨道A (已知职业): {len(descs_known)} 个描述待分类...")
        BATCH_SIZE = 50
        for i in range(0, len(descs_known), BATCH_SIZE):
            batch = descs_known[i:i+BATCH_SIZE]
            # 调用LLM，使用全量Info，无限制
            res = get_standard_category(batch, class_info_full, restricted=False)
            mapping_dict.update(res)

    # 轨道B：Profession 为 "未知" 的样本 -> 归类 Identifier -> 使用限制分类表(0,9,10)
    unknown_mask = df_data['profession'] == '未知'
    descs_unknown = df_data.loc[unknown_mask, 'identifier'].dropna().unique().tolist()
    
    if descs_unknown:
        print(f"  - 轨道B (未知职业): {len(descs_unknown)} 个描述待分类 (限制为非从业类)...")
        BATCH_SIZE = 50
        for i in range(0, len(descs_unknown), BATCH_SIZE):
            batch = descs_unknown[i:i+BATCH_SIZE]
            # 调用LLM，使用限制Info，开启限制模式
            res = get_standard_category(batch, class_info_restricted, restricted=True)
            mapping_dict.update(res)

    # === 回填结果 ===
    def apply_mapping(row):
        # 根据逻辑决定使用哪个字段去查字典
        key = row['identifier'] if row['profession'] == '未知' else row['profession']
        return mapping_dict.get(key, '匹配失败') # 如果没匹配到，返回标记

    df_data['职业分类代码'] = df_data.apply(apply_mapping, axis=1)

    # 映射名称
    code_to_name = df_class.set_index('代码')['分类名称'].to_dict()
    df_data['职业分类名称'] = df_data['职业分类代码'].map(code_to_name)

    # 保存
    df_data.to_excel(output_path, index=False)
    print(f"  - 已保存至: {output_path}")

def main():
    # 1. 准备分类表
    print("加载分类表...")
    df_class = pd.read_excel(CLASS_FILE) if CLASS_FILE.endswith('.xlsx') else pd.read_csv(CLASS_FILE)
    
    # 动态添加第10类（防止Excel里没有）
    if 10 not in df_class['代码'].values:
        print("提示：自动在内存中补充'10-非从业人员-其他'分类")
        new_row = pd.DataFrame([{
            '代码': 10, 
            '分类名称': '非从业人员-其他', 
            '分类内容': '无法归入学生、儿童或家庭角色的其他非从业人员，或身份不明者。'
        }])
        df_class = pd.concat([df_class, new_row], ignore_index=True)

    # 2. 准备输出文件夹
    if not os.path.exists(OUTPUT_FOLDER):
        os.makedirs(OUTPUT_FOLDER)
        print(f"创建输出文件夹: {OUTPUT_FOLDER}")

    # 3. 遍历处理文件
    files = [f for f in os.listdir(INPUT_FOLDER) if f.endswith(('.xlsx', '.xls', '.csv'))]
    print(f"共发现 {len(files)} 个数据文件。")

    for file_name in files:
        input_path = os.path.join(INPUT_FOLDER, file_name)
        output_path = os.path.join(OUTPUT_FOLDER, f"已分类_{file_name.replace('.csv', '.xlsx')}")
        
        process_single_file(input_path, output_path, df_class)
        time.sleep(0.5) # 防止API速率限制

    print("\n所有文件处理完毕！")

if __name__ == "__main__":
    main()